In [158]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier 
import statsmodels.api as sm
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


url = 'https://raw.githubusercontent.com/AntonioNazar/Oracle_Desafio_2/refs/heads/main/telecom_churn.csv'

# 1. Revendo os dados já tratados

In [136]:
df = pd.read_csv(url)
df.head()

,churn,gender,senior_citizen,partner,dependents,tenure,phone_service,multiple_lines,internet_service,online_security,...,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,charges_monthly,charges_total,feature_citizen
0,No,Female,No,Yes,Yes,9,Yes,No,DSL,No,...,No,Yes,Yes,No,One year,Yes,Mailed check,65.6,593.30,0
1,No,Male,No,No,No,9,Yes,Yes,DSL,No,...,No,No,No,Yes,Month-to-month,No,Mailed check,59.9,542.40,0
2,Yes,Male,No,No,No,4,Yes,No,Fiber optic,No,...,Yes,No,No,No,Month-to-month,Yes,Electronic check,73.9,280.85,0
3,Yes,Male,Yes,Yes,No,13,Yes,No,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,98.0,1237.85,1
4,Yes,Female,Yes,Yes,No,3,Yes,No,Fiber optic,No,...,No,Yes,Yes,No,Month-to-month,Yes,Mailed check,83.9,267.40,1


In [137]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   churn              7043 non-null   object 
 1   gender             7043 non-null   object 
 2   senior_citizen     7043 non-null   object 
 3   partner            7043 non-null   object 
 4   dependents         7043 non-null   object 
 5   tenure             7043 non-null   int64  
 6   phone_service      7043 non-null   object 
 7   multiple_lines     7043 non-null   object 
 8   internet_service   7043 non-null   object 
 9   online_security    7043 non-null   object 
 10  online_backup      7043 non-null   object 
 11  device_protection  7043 non-null   object 
 12  tech_support       7043 non-null   object 
 13  streaming_tv       7043 non-null   object 
 14  streaming_movies   7043 non-null   object 
 15  contract           7043 non-null   object 
 16  paperless_billing  7043 

In [138]:
px.histogram(df, x='churn', color='churn', title='Distribuição de Churn')

# 2. Transformando dados categóricos e normalizando dados numéricos

### 2.1 One-hot-encode e label_encode

In [139]:
lista_colunas_categoricas = ['churn', 'gender', 'senior_citizen', 'partner', 'dependents', 'phone_service', 'multiple_lines', 'internet_service',
'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method']

one_hot = make_column_transformer((OneHotEncoder(drop = 'if_binary'), lista_colunas_categoricas), remainder='passthrough', sparse_threshold=0)

x = one_hot.fit(df[lista_colunas_categoricas])

nomes_colunas = one_hot.get_feature_names_out(lista_colunas_categoricas)

df_X = pd.DataFrame(one_hot.transform(df[lista_colunas_categoricas]), columns=nomes_colunas)

df_X.head()

,onehotencoder__churn_Yes,onehotencoder__gender_Male,onehotencoder__senior_citizen_Yes,onehotencoder__partner_Yes,onehotencoder__dependents_Yes,onehotencoder__phone_service_Yes,onehotencoder__multiple_lines_No,onehotencoder__multiple_lines_No phone service,onehotencoder__multiple_lines_Yes,onehotencoder__internet_service_DSL,...,onehotencoder__streaming_movies_No internet service,onehotencoder__streaming_movies_Yes,onehotencoder__contract_Month-to-month,onehotencoder__contract_One year,onehotencoder__contract_Two year,onehotencoder__paperless_billing_Yes,onehotencoder__payment_method_Bank transfer (automatic),onehotencoder__payment_method_Credit card (automatic),onehotencoder__payment_method_Electronic check,onehotencoder__payment_method_Mailed check
0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
1,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


### 2.2 Normalização

In [140]:
lista_colunas_numericas = ['tenure', 'charges_monthly', 'charges_total']

scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(df[lista_colunas_numericas])

X_scaled_df = pd.DataFrame(X_scaled, columns=lista_colunas_numericas)

X_scaled_df.head()

,tenure,charges_monthly,charges_total
0,0.125000,0.471144,0.068315
1,0.125000,0.414428,0.062454
2,0.055556,0.553731,0.032338
3,0.180556,0.793532,0.142531
4,0.041667,0.653234,0.030789


In [141]:
X_final = pd.concat([df_X, X_scaled_df], axis=1).astype(float)

In [142]:
X_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 41 columns):
 #   Column                                                   Non-Null Count  Dtype  
---  ------                                                   --------------  -----  
 0   onehotencoder__churn_Yes                                 7043 non-null   float64
 1   onehotencoder__gender_Male                               7043 non-null   float64
 2   onehotencoder__senior_citizen_Yes                        7043 non-null   float64
 3   onehotencoder__partner_Yes                               7043 non-null   float64
 4   onehotencoder__dependents_Yes                            7043 non-null   float64
 5   onehotencoder__phone_service_Yes                         7043 non-null   float64
 6   onehotencoder__multiple_lines_No                         7043 non-null   float64
 7   onehotencoder__multiple_lines_No phone service           7043 non-null   float64
 8   onehotencoder__multiple_line

# 3. Analisando correlação entre as features

Para analisar a seguinte matriz a seguir, será seguida a seguinte metodologia:
1. Visualizar as features significativas (valor absoluto maior que 0.3) correlação com a variável objetivo (churn)
2. Caso ela seja primeira, escolher ela, caso não, visualizar sua correlação com as outras

In [143]:
X_final.corr()

,onehotencoder__churn_Yes,onehotencoder__gender_Male,onehotencoder__senior_citizen_Yes,onehotencoder__partner_Yes,onehotencoder__dependents_Yes,onehotencoder__phone_service_Yes,onehotencoder__multiple_lines_No,onehotencoder__multiple_lines_No phone service,onehotencoder__multiple_lines_Yes,onehotencoder__internet_service_DSL,...,onehotencoder__contract_One year,onehotencoder__contract_Two year,onehotencoder__paperless_billing_Yes,onehotencoder__payment_method_Bank transfer (automatic),onehotencoder__payment_method_Credit card (automatic),onehotencoder__payment_method_Electronic check,onehotencoder__payment_method_Mailed check,tenure,charges_monthly,charges_total
onehotencoder__churn_Yes,1.000000,-0.008612,0.150889,-0.150448,-0.164221,0.011942,-0.032569,-0.011942,0.040102,-0.124214,...,-0.177820,-0.302253,0.191825,-0.117937,-0.134302,0.301919,-0.091683,-0.352229,0.193356,-0.198324
onehotencoder__gender_Male,-0.008612,1.000000,-0.001874,-0.001808,0.010517,-0.006488,0.004476,0.006488,-0.008414,0.006568,...,0.008026,-0.003695,-0.011754,-0.016024,0.001215,0.000752,0.013744,0.005106,-0.014569,-0.000080
onehotencoder__senior_citizen_Yes,0.150889,-0.001874,1.000000,0.016479,-0.211185,0.008576,-0.136213,-0.008576,0.142948,-0.108322,...,-0.046262,-0.117000,0.156530,-0.016159,-0.024135,0.171718,-0.153477,0.016567,0.220173,0.103006
onehotencoder__partner_Yes,-0.150448,-0.001808,0.016479,1.000000,0.452676,0.017706,-0.129929,-0.017706,0.142057,-0.000851,...,0.082783,0.248091,-0.014877,0.110706,0.082029,-0.083852,-0.095125,0.379697,0.096848,0.317504
onehotencoder__dependents_Yes,-0.164221,0.010517,-0.211185,0.452676,1.000000,-0.001762,0.023198,0.001762,-0.024526,0.052010,...,0.068368,0.204613,-0.111377,0.052021,0.060267,-0.150642,0.059071,0.159712,-0.113890,0.062078
onehotencoder__phone_service_Yes,0.011942,-0.006488,0.008576,0.017706,-0.001762,1.000000,0.315431,-1.000000,0.279690,-0.452425,...,-0.002791,0.003519,0.016505,0.007556,-0.007721,0.003062,-0.003319,0.008448,0.247398,0.113214
onehotencoder__multiple_lines_No,-0.032569,0.004476,-0.136213,-0.129929,0.023198,0.315431,1.000000,-0.315431,-0.822853,-0.070179,...,0.002098,-0.102937,-0.151864,-0.070178,-0.063921,-0.080836,0.222605,-0.323088,-0.338314,-0.396059
onehotencoder__multiple_lines_No phone service,-0.011942,0.006488,-0.008576,-0.017706,0.001762,-1.000000,-0.315431,1.000000,-0.279690,0.452425,...,0.002791,-0.003519,-0.016505,-0.007556,0.007721,-0.003062,0.003319,-0.008448,-0.247398,-0.113214
onehotencoder__multiple_lines_Yes,0.040102,-0.008414,0.142948,0.142057,-0.024526,0.279690,-0.822853,-0.279690,1.000000,-0.199920,...,-0.003794,0.106253,0.163530,0.075527,0.060048,0.083618,-0.227206,0.331941,0.490434,0.468504
onehotencoder__internet_service_DSL,-0.124214,0.006568,-0.108322,-0.000851,0.052010,-0.452425,-0.070179,0.452425,-0.199920,1.000000,...,0.046795,0.031714,-0.063121,0.025476,0.051438,-0.104418,0.041899,0.013274,-0.160189,-0.052469


Foi perceptível que as features foram significativas: 
1. onehotencoder__internet_service_Fiber optic	
2. onehotencoder__online_security_No
3. onehotencoder__tech_support_No
4. onehotencoder__contract_Month-to-month	
5. onehotencoder__contract_Two year
6. onehotencoder__payment_method_Electronic check
7. tenure

Mas é necessário perceber que o contrato mensal e de dois anos tem uma forte correlação, sendo descartada a com menor correlação com a variável alvo, ou seja, a onehotencoder__contract_Two year.

# 4. Treinando modelos

In [144]:
colunas_significativas = [ 'onehotencoder__contract_Month-to-month',
       'tenure', 'onehotencoder__online_security_No',
       'onehotencoder__tech_support_No',
       'onehotencoder__internet_service_Fiber optic',
       'onehotencoder__payment_method_Electronic check']
colunas_significativas

['onehotencoder__contract_Month-to-month',
 'tenure',
 'onehotencoder__online_security_No',
 'onehotencoder__tech_support_No',
 'onehotencoder__internet_service_Fiber optic',
 'onehotencoder__payment_method_Electronic check']

separação do dataset em treino e teste

In [145]:
sm.add_constant(X_final)

,const,onehotencoder__churn_Yes,onehotencoder__gender_Male,onehotencoder__senior_citizen_Yes,onehotencoder__partner_Yes,onehotencoder__dependents_Yes,onehotencoder__phone_service_Yes,onehotencoder__multiple_lines_No,onehotencoder__multiple_lines_No phone service,onehotencoder__multiple_lines_Yes,...,onehotencoder__contract_One year,onehotencoder__contract_Two year,onehotencoder__paperless_billing_Yes,onehotencoder__payment_method_Bank transfer (automatic),onehotencoder__payment_method_Credit card (automatic),onehotencoder__payment_method_Electronic check,onehotencoder__payment_method_Mailed check,tenure,charges_monthly,charges_total
0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.125000,0.471144,0.068315
1,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.125000,0.414428,0.062454
2,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.055556,0.553731,0.032338
3,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.180556,0.793532,0.142531
4,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.041667,0.653234,0.030789
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.180556,0.367164,0.085540
7039,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.305556,0.665174,0.215745
7040,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.027778,0.318905,0.010680
7041,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.930556,0.493532,0.532845


In [146]:
x_treino, x_teste, y_treino, y_teste = train_test_split(X_final[colunas_significativas], X_final['onehotencoder__churn_Yes'], test_size=0.2, random_state=404)

### 4.1 Modelo ingênuo

Serve de base para avaliação dos outros modelos

In [173]:
dummy = DummyClassifier()
dummy.fit(x_treino, y_treino)
dummy.score(x_teste, y_teste)

0.7381121362668559

### 4.2 Regressão Logística

In [180]:
reg_log = sm.Logit(y_treino, x_treino, hasconst = True).fit()

Optimization terminated successfully.
         Current function value: 0.460164
         Iterations 7


In [181]:
print(reg_log.summary())

                              Logit Regression Results                              
Dep. Variable:     onehotencoder__churn_Yes   No. Observations:                 5634
Model:                                Logit   Df Residuals:                     5628
Method:                                 MLE   Df Model:                            5
Date:                      Thu, 05 Mar 2026   Pseudo R-squ.:                  0.2059
Time:                              03:17:58   Log-Likelihood:                -2592.6
converged:                             True   LL-Null:                       -3264.8
Covariance Type:                  nonrobust   LLR p-value:                1.466e-288
                                                     coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------------
onehotencoder__contract_Month-to-month            -0.5664      0.072     -7.885      0.000

In [184]:
y_pred_prob = reg_log.predict(x_treino)
y_pred = (y_pred_prob >= 0.5).astype(int)
accuracy = accuracy_score(y_treino, y_pred)
precision = precision_score(y_treino, y_pred)
recall = recall_score(y_treino, y_pred)
f1 = f1_score(y_treino, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

cm = confusion_matrix(y_treino, y_pred)
print(cm)

Accuracy: 0.7928647497337593
Precision: 0.6374896779521056
Recall: 0.5146666666666667
F1-score: 0.5695315381777941
[[3695  439]
 [ 728  772]]


### 4.3 Árvore de decisão

In [168]:
arvore = DecisionTreeClassifier(random_state=404)
arvore.fit(x_treino, y_treino)
arvore.score(x_teste, y_teste)

0.7707594038325053

In [169]:
y_pred_prob = arvore.predict(x_treino)

accuracy = accuracy_score(y_treino, y_pred_prob)
precision = precision_score(y_treino, y_pred_prob)
recall = recall_score(y_treino, y_pred_prob)
f1 = f1_score(y_treino, y_pred_prob)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

cm = confusion_matrix(y_treino, y_pred_prob)
print(cm)

Accuracy: 0.8422080227192048
Precision: 0.7333842627960275
Recall: 0.64
F1-score: 0.6835172659309363
[[3785  349]
 [ 540  960]]


### 4.4 KNN

In [170]:
knn = KNeighborsClassifier()
knn.fit(x_treino, y_treino)
knn.score(x_teste, y_teste)

0.7814052519517388

In [179]:
y_pred_prob = knn.predict(x_treino)

accuracy = accuracy_score(y_treino, y_pred_prob)
precision = precision_score(y_treino, y_pred_prob)
recall = recall_score(y_treino, y_pred_prob)
f1 = f1_score(y_treino, y_pred_prob)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

cm = confusion_matrix(y_treino, y_pred_prob)
print(cm)

Accuracy: 0.8072417465388712
Precision: 0.6508746355685131
Recall: 0.5953333333333334
F1-score: 0.621866295264624
[[3655  479]
 [ 607  893]]


# 5. Relatório final